# 🤖 KubeLLM — Fine-tuning phi-2 + LoRA
## Contribution scientifique C1 — AI-Kube Master Recherche 2025

**Étapes :**
1. Vérifier le GPU
2. Installer les packages
3. Uploader le dataset
4. Charger phi-2 + LoRA
5. Évaluer AVANT fine-tuning (BLEU baseline)
6. Fine-tuning
7. Évaluer APRÈS fine-tuning (BLEU amélioré)
8. Sauvegarder KubeLLM

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 1 — Vérifier le GPU T4
# ═══════════════════════════════════════════════════════════
import torch

print('=== Environnement GPU ===')
print(f'GPU disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print('✅ GPU OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 2 — Installer les packages compatibles T4
# ═══════════════════════════════════════════════════════════
!pip uninstall -y bitsandbytes peft trl transformers accelerate -q
!pip install -q transformers==4.38.2
!pip install -q peft==0.9.0
!pip install -q trl==0.7.11
!pip install -q accelerate==0.27.2
!pip install -q bitsandbytes==0.41.3
!pip install -q sacrebleu rouge-score
print('✅ Packages installés')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 3 — Uploader le dataset kubernetes_pairs.jsonl
# ═══════════════════════════════════════════════════════════
from google.colab import files
print('Sélectionne kubernetes_pairs.jsonl depuis ton PC...')
uploaded = files.upload()
print(f'✅ Fichier uploadé : {list(uploaded.keys())}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 4 — Préparer le dataset
# ═══════════════════════════════════════════════════════════
import json, yaml
from datasets import Dataset

pairs = []
with open('kubernetes_pairs.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            pairs.append(json.loads(line))
        except:
            pass

print(f'✅ {len(pairs)} paires chargées')

def format_pair(example):
    output = example['output']
    if isinstance(output, dict):
        try:
            output = yaml.dump(output, default_flow_style=False, allow_unicode=True)
        except:
            output = str(output)
    elif not isinstance(output, str):
        output = str(output)

    inp = example.get('input', 'Cluster Kubernetes standard')
    if isinstance(inp, dict):
        inp = str(inp)

    return {"text": f"""### Instruction:\n{example['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n{output}"""}

formatted = [format_pair(p) for p in pairs]
dataset = Dataset.from_list(formatted)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f'✅ Train : {len(dataset["train"])} exemples')
print(f'   Test  : {len(dataset["test"])} exemples')
print(f'\nExemple (300 chars) :')
print(dataset['train'][0]['text'][:300])

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 5 — Charger phi-2 + LoRA
# ═══════════════════════════════════════════════════════════
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'microsoft/phi-2'

print('🔄 Chargement tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('🔄 Chargement phi-2 en float16 (3-5 min)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

print('🔄 Application LoRA rank=8...')
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'dense'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ Modèle prêt !')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 6 — Évaluation AVANT fine-tuning (BLEU baseline)
# C'est la mesure scientifique de référence pour C1
# ═══════════════════════════════════════════════════════════
import sacrebleu

def generate_yaml(model, tokenizer, instruction, max_new_tokens=200):
    prompt = f"""### Instruction:\n{instruction}\n\n### Input:\nCluster Kubernetes standard\n\n### Response:\n"""
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extraire seulement la réponse
    if '### Response:' in generated:
        generated = generated.split('### Response:')[-1].strip()
    return generated

print('📊 Évaluation BLEU avant fine-tuning (baseline phi-2)...')
print('   (teste sur 5 exemples du set de test)')

test_samples = dataset['test'].select(range(min(5, len(dataset['test']))))
references = []
hypotheses = []

for i, sample in enumerate(test_samples):
    # Extraire instruction et référence depuis le texte formaté
    parts = sample['text'].split('### Response:')
    if len(parts) >= 2:
        instruction_part = parts[0].replace('### Instruction:', '').replace('### Input:', '').strip()
        reference = parts[1].strip()
        instruction = instruction_part.split('\n')[0].strip()

        generated = generate_yaml(model, tokenizer, instruction)
        references.append([reference])
        hypotheses.append(generated)
        print(f'  [{i+1}/5] ✅ Généré {len(generated)} chars')

# Calcul BLEU
bleu = sacrebleu.corpus_bleu(hypotheses, [[r[0] for r in references]])
BLEU_BEFORE = bleu.score
print(f'\n📊 BLEU score AVANT fine-tuning : {BLEU_BEFORE:.2f}')
print('   (sera comparé avec le score APRÈS pour mesurer l\'amélioration)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 7 — Fine-tuning avec SFTTrainer
# Durée estimée : 20-40 min sur T4
# ═══════════════════════════════════════════════════════════
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='./kubellm-phi2-lora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    warmup_steps=10,
    lr_scheduler_type='cosine',
    report_to='none',
    optim='adamw_torch',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print('🚀 Début du fine-tuning KubeLLM...')
print(f'   Dataset : {len(dataset["train"])} exemples')
print(f'   Epochs  : 3')
print(f'   Modèle  : phi-2 + LoRA rank=8')
print('   Durée estimée : 20-40 min sur T4')
print()

trainer.train()

print('\n✅ Fine-tuning terminé !')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 8 — Évaluation APRÈS fine-tuning
# Mesure l'amélioration BLEU — donnée scientifique clé pour C1
# ═══════════════════════════════════════════════════════════
print('📊 Évaluation BLEU après fine-tuning (KubeLLM)...')

hypotheses_after = []
for i, sample in enumerate(test_samples):
    parts = sample['text'].split('### Response:')
    if len(parts) >= 2:
        instruction_part = parts[0].replace('### Instruction:', '').replace('### Input:', '').strip()
        instruction = instruction_part.split('\n')[0].strip()
        generated = generate_yaml(model, tokenizer, instruction)
        hypotheses_after.append(generated)
        print(f'  [{i+1}/5] ✅ Généré {len(generated)} chars')

bleu_after = sacrebleu.corpus_bleu(hypotheses_after, [[r[0] for r in references]])
BLEU_AFTER = bleu_after.score
improvement = ((BLEU_AFTER - BLEU_BEFORE) / max(BLEU_BEFORE, 0.1)) * 100

print(f'\n' + '='*50)
print(f'RÉSULTATS SCIENTIFIQUES — Contribution C1')
print(f'='*50)
print(f'BLEU phi-2 base    : {BLEU_BEFORE:.2f}')
print(f'BLEU KubeLLM       : {BLEU_AFTER:.2f}')
print(f'Amélioration       : +{improvement:.1f}%')
print(f'Modèle             : microsoft/phi-2 + LoRA rank=8')
print(f'Dataset            : {len(pairs)} paires Kubernetes')
print(f'='*50)
print(f'\n✅ Ces métriques vont dans ton mémoire de recherche !')

In [ ]:
# ═══════════════════════════════════════════════════════════
# ÉTAPE 9 — Sauvegarder KubeLLM
# ═══════════════════════════════════════════════════════════
import os
from google.colab import files

SAVE_PATH = './kubellm-v1'

print('💾 Sauvegarde du modèle KubeLLM...')
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# Sauvegarder les métriques
import json
metrics = {
    'model': 'KubeLLM-v1',
    'base_model': 'microsoft/phi-2',
    'method': 'LoRA rank=8',
    'dataset_size': len(pairs),
    'epochs': 3,
    'bleu_before': round(BLEU_BEFORE, 2),
    'bleu_after': round(BLEU_AFTER, 2),
    'bleu_improvement_pct': round(improvement, 1),
}
with open(f'{SAVE_PATH}/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'✅ Modèle sauvegardé dans {SAVE_PATH}/')

# Zipper pour télécharger
!zip -r kubellm-v1.zip kubellm-v1/
print('✅ ZIP créé — téléchargement...')
files.download('kubellm-v1.zip')
print('✅ KubeLLM téléchargé sur ton PC !')

In [ ]:
# ═══════════════════════════════════════════════════════════
# OPTIONNEL — Test final de KubeLLM
# ═══════════════════════════════════════════════════════════
test_prompts = [
    'Déployer nginx avec 3 replicas et limite 512Mi RAM',
    'Créer un HPA pour flask-api: min 2 max 8 replicas si CPU>70%',
    'NetworkPolicy deny-all pour namespace production',
]

print('🧪 Test de KubeLLM sur 3 prompts Kubernetes')
print('='*60)
for prompt in test_prompts:
    print(f'\n📝 Prompt: {prompt}')
    result = generate_yaml(model, tokenizer, prompt, max_new_tokens=300)
    print(f'🤖 KubeLLM:\n{result[:500]}')
    print('-'*40)